<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-02-model-adapter/notebook.ipynb)


# Session 2 — Call a model through the adapter

**Goal:** call a model through one method, run the same prompt on two lanes, and turn a provider failure into a refusal instead of a traceback. *Thread: harness engineering.*

Everything runs on the deterministic `FakeLLM` by default. `LIVE` is whichever lane your `.env` names; when that lane is not reachable the preflight says so and `LIVE` is a `FakeLLM` too. No cell here needs a key, a network, or a paid call.

In [42]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.13 (need >= 3.11)
⚠️  kernel is the repo .venv  -> pick the .venv kernel in Jupyter, or start it with: uv run jupyter lab
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [44]:
from bootcamp_agent.checks import check, review

## 1. One call through the seam

A system instruction, a user question, a text answer. `complete(system, user) -> str` is the whole boundary, and every framework wraps it. `FakeLLM` answers from a keyword table, so the same question always gets the same answer.

In [49]:
from bootcamp_agent.llm import FakeLLM

hello_llm = FakeLLM(
    responses={
        "hello": "Hello! I am a deterministic stand-in for a language model.",
        "agent": "An agent is a loop around a model: perceive, decide, act, observe.",
    },
    default="I have no canned answer for that — a real model would improvise here.",
)

print(hello_llm.complete(system="You are concise.", user="Say hello to the bootcamp"))

Hello! I am a deterministic stand-in for a language model.


## 2. Exercise: three questions, three paths

**Context.** `hello_llm` knows two keywords, `agent` and `hello`, matched
case-insensitively **in insertion order**. Anything else gets the default. Three
calls show all three paths.

**What the cell below already does.** It asks three questions — one containing
`agent`, one containing `hello`, one containing neither — and prints all three
replies. It passes `ch02-e1` as it stands.

**Run it, and read the output before anything else.** Three lines, three
different answers, and the third is the default. That is the whole behaviour.

**Now improve it.** The rule says *insertion order* decides which keyword wins.
Nothing here proves it. Ask a question containing **both** and see:

```python
print(hello_llm.complete(system="You are concise.", user="hello, what is an agent?"))
```

`agent` was inserted first, so `agent` wins. You have just established by
experiment something the docstring only claimed — which is the habit this whole
course is about.

**Use a separate `print`, not a fourth `answers.append`.** `ch02-e1` requires
exactly three replies, and a fourth would fail a check that was passing. Reading
what a check actually demands before changing the code it grades is the same
skill, one level up.

In [54]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# To stand above it: prove which keyword wins when a question contains both —
# with a separate print, because this list must stay at exactly three.
# Change it, re-run the check cell below, and keep what you learn.
# ---------------------------------------------------------------------
answers = []
answers.append(hello_llm.complete(system="You are concise.", user="What is an agent?"))  # case 1, done
answers.append(hello_llm.complete(system="You are concise.", user="hello there"))
answers.append(hello_llm.complete(system="You are concise.", user="What is the moon made of?"))

for reply in answers:
    print(reply)
print(
    hello_llm.complete(
        system="You are concise.",
        user="hello, what is an agent?"
        )
)

An agent is a loop around a model: perceive, decide, act, observe.
Hello! I am a deterministic stand-in for a language model.
I have no canned answer for that — a real model would improvise here.
Hello! I am a deterministic stand-in for a language model.


**Expected output** (yours may differ in wording, not in shape):

```
An agent is a loop around a model: perceive, decide, act, observe.
Hello! I am a deterministic stand-in for a language model.
I have no canned answer for that — a real model would improvise here.
✅ ch02-e1 passed
```

In [55]:
check("ch02-e1", answers)

✅ ch02-e1 passed


True

## 3. The seam itself

`FakeLLM`, `OllamaClient`, `AnthropicClient` and `OpenAICompatibleClient` satisfy the same one-method protocol. Nothing inherits from it: a class is an `LLMClient` because it has `complete(system, user) -> str`. Swapping providers changes no application code. `LIVE`, from the preflight cell, is whichever lane this machine could actually reach.

In [56]:
import inspect

from bootcamp_agent import llm

print(inspect.getsource(llm.LLMClient))
print(f"LIVE is a {type(LIVE).__name__}")

class LLMClient(Protocol):
    def complete(self, system: str, user: str) -> str:
        """Return the model's text for one system+user exchange."""
        ...

LIVE is a FakeLLM


## 4. Exercise: the same prompt, two lanes

**Context.** The fake is deterministic: same question, same answer, every time.
A real model is not. That gap is why week 2 spends a whole session on
evaluation — you measure properties, not words.

**This cell needs no provider.** Without one, `LIVE` is a `FakeLLM`, the live
pair comes back identical, and that is a **correct run, not a skipped one**. Its
default answer is the course's refusal JSON; session 3 is where that string
stops being text and becomes a parsed contract.

**What the cell below already does.** It asks the same question twice on each
lane and prints whether the two answers agreed. It passes `ch02-e2` as it
stands.

**Run it and read the two lines.** On `fake` both pairs agree. On the ollama
lane the live pair usually does not. Both outcomes are correct — the point is
that you can *see* which lane you are on without being told.

**Now improve it.** `same answer twice?` is a boolean, and a boolean cannot tell
"one word moved" from "a completely different answer". Measure the gap instead:

```python
import difflib

for lane, pair in runs.items():
    ratio = difflib.SequenceMatcher(None, pair[0], pair[1]).ratio()
    print(f"[{lane}] similarity {ratio:.2f}")
```

`1.00` on `fake`, every time. Something lower on a real lane — and *how much*
lower is the number week 2 teaches you to put a threshold on.

**Keep `runs` at exactly two replies per lane.** `ch02-e2` requires it, so do
the extra measuring *after* the dict is built, not by making it bigger. Read
what a check demands before you change the code it grades.

**Want a real second lane?** `BOOTCAMP_PROVIDER=ollama` in `.env`. See
[a local model](../../unit0/local-model.mdx). Nothing here needs it.

In [60]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# To stand above it: measure the difference between the live replies instead of
# printing a boolean — `runs` itself must stay at exactly two per lane.
# Change it, re-run the check cell below, and keep what you learn.
# ---------------------------------------------------------------------
import difflib

question = "Explain in one sentence what an agent is."

runs = {
    "fake": [hello_llm.complete(system="You are concise.", user=question) for _ in range(2)],  # done
    "live": [LIVE.complete(system="You are concise.", user=question) for _ in range(2)],
}

for lane, pair in runs.items():
    ratio = difflib.SequenceMatcher(None, pair [0], pair[1]).ratio()
    print(f"[{lane}] similarity {ratio:.2f}")

[fake] similarity 1.00
[live] similarity 1.00


**Expected output** (yours may differ in wording, not in shape):

```
[fake] same answer twice? True
   An agent is a loop around a model: perceive, decide, act, observe.
   An agent is a loop around a model: perceive, decide, act, observe.
[live] same answer twice? False        <- True on the fake lane, usually False on ollama
   An agent is a program that uses a model to decide which actions to take toward a goal.
   An agent is software that plans, calls tools, and checks results until a task is done.
✅ ch02-e2 passed
```

In [61]:
check("ch02-e2", runs)

✅ ch02-e2 passed


True

## 5. Exercise: your reliability bar

**Context.** Reliability is a decision you make **before** the first real
answer, not after. Write yours down now so the evaluation week has a target that
is yours rather than one we chose.

**What the cell below already does.** It carries three example sentences and
prints them. It passes `ch02-e3` as it stands — the check only measures that
each sentence is written and complete, never that it agrees with ours.

**Now improve it, and this is the one where shipped text is worth the least.**
Replace all three with your own. The test of a good sentence is whether you
could check it tomorrow:

| Too vague | Checkable |
|---|---|
| "when the answer is good" | "when it cites a document I can open" |
| "for important things" | "when the answer would change money or credentials" |
| "risky operations" | "delete a branch, rotate a key, send an email" |

A sentence naming a *category* is a sentence you will argue about later. A
sentence naming a *thing that happens* is one you can act on.

In [64]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# To stand above it: make every sentence name a thing you could actually check tomorrow, not a category.
# Change it, re-run the check cell below, and keep what you learn.
# ---------------------------------------------------------------------
reliability_bar = {
    "reliable_when": "the answer is supported by a source I can verify, or the system clearlyy sats it does not know.",
    "review_when": "the answer could effect money, credentials, production changes, or another consequential decision.",
    "never_unreviewed": "send messages, change shared code, delete data, or modify credentials",
}
for key, sentence in reliability_bar.items():
    print(f"{key:18} {sentence}")

reliable_when      the answer is supported by a source I can verify, or the system clearlyy sats it does not know.
review_when        the answer could effect money, credentials, production changes, or another consequential decision.
never_unreviewed   send messages, change shared code, delete data, or modify credentials


**Expected output** (yours may differ in wording, not in shape):

```
reliable_when      it cites a corpus document I can open, or plainly says it does not know.
review_when        the answer would change money, credentials, or anything in production.
never_unreviewed   rotate a secret or delete a branch on the shared repository.
✅ ch02-e3 passed
```

In [65]:
check("ch02-e3", reliability_bar)

✅ ch02-e3 passed


True

## 6. Failure injection: the two configuration failures

**Not scored, and run it anyway.** Both failures happen inside your process, before any request leaves the machine, so they cost nothing and are the same on every lane.

Watch two things in the output: the missing-key message names `.env.example` and never the key, and the unknown-provider message lists the lanes that do exist. An error that carries its own fix is the standard for the rest of the course.

In [66]:
from bootcamp_agent.config import ConfigError, Settings, load_settings
from bootcamp_agent.llm import get_client

# 1. A lane that needs a key, with no key in the environment.
try:
    get_client(Settings(provider="anthropic", model=None, api_key=None, base_url=None))
except ConfigError as error:
    print("missing credential ->", error)

# 2. A provider name this course does not serve.
try:
    load_settings(env={"BOOTCAMP_PROVIDER": "gpt-9"})
except ConfigError as error:
    print("unsupported lane   ->", error)

missing credential -> Provider 'anthropic' needs an API key in the environment; see .env.example. (The key itself is never printed.)
unsupported lane   -> Unknown BOOTCAMP_PROVIDER 'gpt-9'; expected one of ('fake', 'ollama', 'anthropic', 'openai')


An unsupported **model** is the third case, and it cannot be answered in your process: only the provider knows what it serves. The local lane asks it, without sending a prompt.

The next cell guards itself. Without `BOOTCAMP_PROVIDER=ollama` it prints a skip line and moves on.

In [67]:
import os

if os.environ.get("BOOTCAMP_PROVIDER", "fake") != "ollama":
    print("skipped: no local lane configured (BOOTCAMP_PROVIDER is not 'ollama')")
else:
    from bootcamp_agent.ollama import DEFAULT_MODEL, probe

    result = probe(model=DEFAULT_MODEL)
    print(f"reachable={result.reachable} model_present={result.model_present}")
    print(result.fix or f"{DEFAULT_MODEL} is pulled and ready")

skipped: no local lane configured (BOOTCAMP_PROVIDER is not 'ollama')


## 7. Exercise: a deadline that becomes a refusal

**This is the one you write.** The other three exercises run as shipped and earn
you **300 of 400**. This cell is the remaining 100, and it is the session's whole
point, which is why it is not done for you.

**Context.** A provider that never answers is the failure every production agent
meets. `urllib` raises `TimeoutError` when the deadline passes; `OllamaClient`
catches that and raises `OllamaError` — which is why you catch both. The caller
should get **one shape either way**, so the timeout leaves as a value, not as an
exception.

**What you are given.** `TimeoutLLM` stands in for any provider that runs past
its deadline: its `complete` always raises.

**What to write.** `answer_with_timeout(question)`:

1. Call `client.complete(system=..., user=question)`
2. Catch `TimeoutError` **and** `OllamaError`
3. Return an `AgentResult` whose answer says the model did not respond, cites
   nothing, has confidence `0.0`, and sets `needs_human_review=True`
4. Put the caught error in the **trace**

A worked shape, if you want the shove:

```python
    try:
        text = client.complete(system="You are concise.", user=question)
    except (TimeoutError, OllamaError) as error:
        return AgentResult(
            answer=ResearchAnswer(
                answer="The model did not respond.",
                citations=(), confidence=0.0, needs_human_review=True,
            ),
            trace=(TraceEvent(kind="llm_call", detail=str(error)),),
        )
```

**Keep the cause out of the answer.** The adapter cannot tell "too slow" from
"no server at all", so do not claim which it was. Saying less than you know is
not weakness here — a refusal that guesses a cause is one nobody can debug.

**Run the cell. The test is that it prints and does not raise.** Then run the
check.

In [76]:
# ---------------------------------------------------------------------
# THE ONLY CELL IN THIS SESSION THAT DOES NOT RUN AS SHIPPED.
# The other three are written: run them and you have 300 of 400 marks.
# This one is the 400. It is the session's point, so it is the one you write.
# ---------------------------------------------------------------------
import time
from bootcamp_agent.agent import AgentResult, TraceEvent
from bootcamp_agent.ollama import OllamaError
from bootcamp_agent.schema import ResearchAnswer


class TimeoutLLM:
    """A provider that never answers: the deadline passes and the transport raises."""

    def complete(self, system: str, user: str) -> str:
        raise TimeoutError("no response within the deadline")


def answer_with_timeout(question: str) -> AgentResult:
    client = TimeoutLLM()
    started = time.monotonic()
    try:
        text = client.complete(
            system="You are concise.",
            user=question
            )
        return AgentResult(
            answer=ResearchAnswer(
                answer=text,
                citations=(),
                confidence=1.0,
                needs_human_review=False
            ),
            trace=(
                TraceEvent(
                    kind="llm_call",
                    detail="Model respinded successfully.",
                ),
            ),
        )
    except (TimeoutError, OllamaError) as error:
        elapsed = time.monotonic() - started

        return AgentResult(
            answer=ResearchAnswer(
                answer="The model did not respond. Please try again.",
                citations=(),
                confidence=0.0,
                needs_human_review=True
            ),
            trace=(TraceEvent(
                kind="llm_call",
                detail=(
                    f"provider=TimeoutLLM; "
                    f"elapsed={elapsed:3f}s; "
                    f"error={error}"
                ),
            ),
            ),
            )

    raise NotImplementedError


try:
    result = answer_with_timeout("How does chunking work in RAG?")
    print("DEFENDED —", result.answer.answer)
except NotImplementedError:
    print("not written yet")

DEFENDED — The model did not respond. Please try again.


**Expected output** (yours may differ in wording, not in shape):

```
DEFENDED — The model did not respond in time.
✅ ch02-e4 passed
```

In [72]:
check("ch02-e4", answer_with_timeout)

✅ ch02-e4 passed


True

## 8. Bonus: a refusal you can actually debug

**Optional, and above full marks.** Section 7 takes you to 400 of 400. This one
is worth nothing and is excluded from every total — `ch02` is four exercises
whether you do it or not.

**The problem with the refusal you just wrote.** It is correct, and a call that
died instantly and one that died after thirty seconds produce **the same two
lines**. Nobody can act on that.

**What this asks for.** Two facts, in the `trace` and never in the answer:

1. **Which lane failed** — name the provider in a `TraceEvent` detail
2. **How long it waited** — measure it with `time.monotonic()`

**And the trap it will refuse.** Do not write "the model timed out" in the
*answer*. The adapter cannot tell "too slow" from "no server at all", and a
refusal that guesses a cause is a refusal nobody can debug. Say less than you
know, in the place where the reader is looking for it.

Edit `answer_with_timeout` above, then run the cell below.

In [77]:
from bootcamp_agent.bonus import bonus

# Ships FAILING, on purpose. There is nothing to earn in a cell that arrives
# green — and the message below names exactly what is missing.
bonus("ch02", answer_with_timeout)

✅ bonus ch02 passed — above the floor.


True

## Exit ticket

- What works? What is unclear? What is your next action?
- Homework: run the same prompt on a second lane, and write one sentence on what changed and one on what did not. Then pick the deadline you would give a chat answer, and say what your code does when it passes.

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [78]:
review("ch02")

ch02: 4/4 passed  ·  400/400 marks


True